In [2]:
from sklearn import svm

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler as  StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import KFold, cross_validate, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from imblearn.pipeline import Pipeline as im_Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_curve, confusion_matrix

In [3]:
df1 = pd.read_csv("master_tripel_cramps_dataset.csv")

df_clean_og = df1.dropna(subset=[
    "cramps",
    "cramps binary",
    "cramps group",
    "phase",
    "lh",
    "estrogen",
    "nightly_temperature",
    "hr_mean",
    "glucose_median",
    "glucose_std"
]).copy()

cramps_map_menstrual = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5
}

cramps_map_binary_menstrual = {
    "Low Pain": 0,
    "High Pain": 1,
}

cramps_map_group_menstrual = {
    "No Pain": 0,
    "Low Pain": 1,
    "High Pain": 2,
}

df_clean_og["cramps"] = df_clean_og["cramps"].map(cramps_map_menstrual)
df_clean_og["cramps binary"] = df_clean_og["cramps binary"].map(cramps_map_binary_menstrual)
df_clean_og["cramps group"] = df_clean_og["cramps group"].map(cramps_map_group_menstrual)

df_clean_og = df_clean_og.dropna(subset=["cramps binary"]).copy()
drop_cols = [
    "id",
    "cramps",
    "cramps binary",
    "cramps group",
    "study_interval_x",
    "study_interval_y",
    "day_in_study"
]

X = df_clean_og.drop(columns=drop_cols)
y = df_clean_og["cramps"].astype(int)

variables_categoricas = ["phase"]
variables_num = [
    col for col in X.columns
    if col not in variables_categoricas]

In [4]:
groups = df_clean_og["id"]

cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_inner = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

import pickle 
with open("outer_splits_binary.pkl", "rb") as f:
    outer_splits = pickle.load(f)

def get_best_rho(y_train, y_train_proba_oof, rhos=np.linspace(0.1, 0.9, 17)):

    best_rho = None
    best_f1 = -1

    for rho in rhos:

        y_pred = (y_train_proba_oof >= rho).astype(int)

        f1 = f1_score(y_train, y_pred, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_rho = rho

    return best_rho

In [5]:
param_grid = {"KNN": {
                    "classifier__n_neighbors": [3, 5, 7, 9, 11],
                    "classifier__weights": ["uniform", "distance"],
                    "classifier__metric": ["euclidean", "manhattan"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}, 
              "LogReg": {
                    "classifier__C": [0.001, 0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "classifier__penalty": ["l1", "l2"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]},
              "Random_Forest": {
                    "classifier__n_estimators": [10, 20, 50, 100, 200, 250, 300, 500],
                    "classifier__max_depth": [1, 2, 3, 5, 7, 10],
                    "classifier__min_samples_leaf": [1, 5, 10],
                    "classifier__max_features": ["sqrt", "log2"],
                    "classifier__max_leaf_nodes": [2, 4, 8, 16, 32],
                    "classifier__class_weight": ["balanced"]},
              "SVM":{
                    "classifier__kernel": ["linear","rbf"],
                    "classifier__gamma": ["scale", "auto"],
                    "classifier__C": [0.01, 0.1, 1, 10],
                    "classifier__class_weight": ["balanced"],
                    "feature_selection__k": [5, 10, 15, 20, "all"]}

    }

In [8]:
model_df = []
type_response_df = []
accuracy_model = []
balanced_accuracy_model = []
f1_macro_model = []
f1_weighted_model = []
models = ["KNN", "LogReg","Random_Forest","SVM"]


accuracy = np.nan * np.ones((len(models), len(outer_splits)))
balanced_accuracy = np.nan * np.ones((len(models), len(outer_splits)))
f1_macro = np.nan * np.ones((len(models), len(outer_splits)))
f1_weighted = np.nan * np.ones((len(models), len(outer_splits)))

best_params_all = {model: [] for model in models}
selected_features_all = {model: [] for model in models}

for j, (train_idx, test_idx) in enumerate(outer_splits):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = groups.iloc[train_idx]
        
    for (i,model) in enumerate(models):
        preprocessor = ColumnTransformer(
            transformers=[
                ("zscore", StandardScaler(), variables_num),
                ("ohe", OneHotEncoder(handle_unknown="ignore"), variables_categoricas)
        ])
        match model:
            case "LogReg":
                mdl = LogisticRegression( solver = "saga", max_iter=1000, random_state = 42)   
            case "Random_Forest":
                mdl = RandomForestClassifier(random_state = 42)
            case "KNN":
                mdl = KNeighborsClassifier()
            case "SVM":
                mdl = svm.SVC(probability = True, random_state = 42)
        if model == "Random_Forest":
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("classifier", mdl)])
        else:
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("feature_selection", SelectKBest(score_func=f_classif)),
                ("classifier", mdl)])

        grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid[model],
                cv=cv_inner,
                scoring="f1_macro",
                n_jobs=-1)
        
        grid.fit(X_train, y_train, groups=groups_train)
        
        mdl_selected = grid.best_estimator_
        best_params_all[model].append(grid.best_params_)
        
        if model != "Random_Forest":
            feature_names = mdl_selected.named_steps["preprocessor"].get_feature_names_out()
            selector = mdl_selected.named_steps["feature_selection"]
            selected_features = feature_names[selector.get_support()].tolist()
            print(selected_features)
        else:
            selected_features = X_train.columns.tolist()

        selected_features_all[model].append(selected_features)

        mdl_selected.fit(X_train, y_train)
        y_pred = mdl_selected.predict(X_test)
        y_prob = mdl_selected.predict_proba(X_test)[:, 1]
        
        accuracy[i, j] = accuracy_score(y_test, y_pred)
        balanced_accuracy[i, j] = balanced_accuracy_score(y_test, y_pred)
        f1_macro[i, j] = f1_score(y_test, y_pred, average="macro")
        f1_weighted[i, j] = f1_score(y_test, y_pred, average="weighted")

        cm = confusion_matrix(y_test, y_pred)

        print(f"\n===== Fold {j+1} - {model} - {grid.best_params_}) =====")
        print(f"Accuracy: {accuracy[i,j]:.4f}")
        print(f"Balanced Accuracy: {balanced_accuracy[i,j]:.4f}")
        print(f"F1 Macro: {f1_macro[i,j]:.4f}")

        print("\nConfusion matrix:")
        print(cm)

        print("\nClassification report:")
        print(classification_report(y_test, y_pred))
        
for (i,model) in enumerate(models):
    accuracy_model.append(f"{np.mean(accuracy[i,:]):.2f} +- {np.std(accuracy[i,:]):.2f} [{np.min(accuracy[i,:]):.2f} -  {np.max(accuracy[i,:]):.2f}]")
    balanced_accuracy_model.append(f"{np.mean(balanced_accuracy[i,:]):.2f} +- {np.std(balanced_accuracy[i,:]):.2f} [{np.min(balanced_accuracy[i,:]):.2f} -  {np.max(balanced_accuracy[i,:]):.2f}]")
    f1_macro_model.append(f"{np.mean(f1_macro[i,:]):.2f} +- {np.std(f1_macro[i,:]):.2f}  [{np.min(f1_macro[i,:]):.2f}  -   {np.max(f1_macro[i,:]):.2f}]")
    f1_weighted_model.append(f"{np.mean(f1_weighted[i,:]):.2f} +- {np.std(f1_weighted[i,:]):.2f}  [{np.min(f1_weighted[i,:]):.2f}  -   {np.max(f1_weighted[i,:]):.2f}]")
        
    if i == 0:
        type_response_df.append("cramps")
    else:
        type_response_df.append("")
        
    model_df.append(model)


df_performance_models = pd.DataFrame(np.transpose([type_response_df,model_df,accuracy_model,balanced_accuracy_model,f1_macro_model,f1_weighted_model]), columns=["Type response","Model", "Accuracy","Balanced Accuracy","F1-Score (macro)","F1-Score (weighted)"])
display(df_performance_models)

['zscore__hr_mean', 'zscore__hr_min', 'zscore__hr_q_05', 'zscore__hr_q_25', 'ohe__phase_Menstrual']

===== Fold 1 - KNN - {'classifier__metric': 'euclidean', 'classifier__n_neighbors': 7, 'classifier__weights': 'distance', 'feature_selection__k': 5}) =====
Accuracy: 0.4479
Balanced Accuracy: 0.1628
F1 Macro: 0.1575

Confusion matrix:
[[207 107  15   6   1   1]
 [ 74  36   6  23   1   0]
 [ 18  11   0   4   1   0]
 [  6   7   2   2   2   0]
 [  5   4   0   1   0   1]
 [  2   2   0   0   2   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.66      0.61      0.64       337
           1       0.22      0.26      0.23       140
           2       0.00      0.00      0.00        34
           3       0.06      0.11      0.07        19
           4       0.00      0.00      0.00        11
           5       0.00      0.00      0.00         6

    accuracy                           0.45       547
   macro avg       0.16      0.16      0.16

C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['zscore__hr_mean', 'zscore__hr_min', 'zscore__hr_q_05', 'zscore__hr_q_25', 'ohe__phase_Menstrual']

===== Fold 2 - SVM - {'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'scale', 'classifier__kernel': 'linear', 'feature_selection__k': 5}) =====
Accuracy: 0.3754
Balanced Accuracy: 0.2390
F1 Macro: 0.1758

Confusion matrix:
[[199   0   5   2   8   2]
 [134   0   4  43  11  13]
 [ 66   0   2  12   4   5]
 [ 20   0   4  18  10   8]
 [  3   0   3   3   4   8]
 [  0   0   1   0   2   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.47      0.92      0.62       216
           1       0.00      0.00      0.00       205
           2       0.11      0.02      0.04        89
           3       0.23      0.30      0.26        60
           4       0.10      0.19      0.13        21
           5       0.00      0.00      0.00         3

    accuracy                           0.38       594
   macro avg       

C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\marti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['zscore__hr_mean', 'zscore__hr_min', 'zscore__hr_q_05', 'zscore__hr_q_25', 'ohe__phase_Menstrual']

===== Fold 3 - KNN - {'classifier__metric': 'manhattan', 'classifier__n_neighbors': 3, 'classifier__weights': 'distance', 'feature_selection__k': 5}) =====
Accuracy: 0.3794
Balanced Accuracy: 0.1953
F1 Macro: 0.1906

Confusion matrix:
[[103  46  13  11   8   4]
 [ 33  18   3   1   0   1]
 [ 13   8   3   0   1   0]
 [ 20  12   5   2   2   0]
 [  3  13   2   3   3   0]
 [  2   6   0   1   0   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.59      0.56      0.57       185
           1       0.17      0.32      0.23        56
           2       0.12      0.12      0.12        25
           3       0.11      0.05      0.07        41
           4       0.21      0.12      0.16        24
           5       0.00      0.00      0.00         9

    accuracy                           0.38       340
   macro avg       0.20      0.20      0.19

C:\Users\marti\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


['zscore__lh', 'zscore__estrogen', 'zscore__nightly_temperature', 'zscore__hr_mean', 'zscore__hr_median', 'zscore__hr_std', 'zscore__hr_cv', 'zscore__hr_var', 'zscore__hr_min', 'zscore__hr_max', 'zscore__hr_q_05', 'zscore__hr_q_25', 'zscore__hr_q_75', 'zscore__hr_q_95', 'zscore__glucose_mean', 'zscore__glucose_median', 'zscore__glucose_std', 'zscore__glucose_cv', 'zscore__glucose_var', 'zscore__glucose_min', 'zscore__glucose_max', 'zscore__glucose_q_05', 'zscore__glucose_q_25', 'zscore__glucose_q_75', 'zscore__glucose_q_95', 'ohe__phase_Fertility', 'ohe__phase_Follicular', 'ohe__phase_Luteal', 'ohe__phase_Menstrual']


C:\Users\marti\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



===== Fold 5 - LogReg - {'classifier__C': 10, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'feature_selection__k': 'all'}) =====
Accuracy: 0.2121
Balanced Accuracy: 0.1688
F1 Macro: 0.1481

Confusion matrix:
[[ 74 122  43   6  18  25]
 [ 17  21  14  16  14   9]
 [  5   6   0   5   5   7]
 [  9  37   8  11  28  11]
 [  2   3   0   4  13   9]
 [  3   4   0   4   8   0]]

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.26      0.37       288
           1       0.11      0.23      0.15        91
           2       0.00      0.00      0.00        28
           3       0.24      0.11      0.15       104
           4       0.15      0.42      0.22        31
           5       0.00      0.00      0.00        19

    accuracy                           0.21       561
   macro avg       0.20      0.17      0.15       561
weighted avg       0.42      0.21      0.25       561


===== Fold 5 - Random_Forest - {'cla

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (7,) + inhomogeneous part.

In [9]:
df_performance_models = pd.DataFrame(np.transpose([type_response_df,model_df,accuracy_model,balanced_accuracy_model,f1_macro_model,f1_weighted_model]), columns=["Type response","Model", "Accuracy","Balanced Accuracy","F1-Score (macro)","F1-Score (weighted)"])
display(df_performance_models)

,Type response,Model,Accuracy,Balanced Accuracy,F1-Score (macro),F1-Score (weighted)
0,cramps,KNN,0.36 +- 0.06 [0.28 - 0.45],0.19 +- 0.02 [0.16 - 0.21],0.18 +- 0.01 [0.16 - 0.19],0.34 +- 0.07 [0.25 - 0.46]
1,,LogReg,0.35 +- 0.10 [0.21 - 0.49],0.26 +- 0.05 [0.17 - 0.31],0.19 +- 0.03 [0.15 - 0.23],0.34 +- 0.10 [0.25 - 0.50]
2,,Random_Forest,0.32 +- 0.05 [0.23 - 0.38],0.22 +- 0.02 [0.20 - 0.24],0.17 +- 0.02 [0.13 - 0.20],0.30 +- 0.07 [0.20 - 0.39]
3,,SVM,0.32 +- 0.07 [0.23 - 0.41],0.25 +- 0.05 [0.17 - 0.32],0.20 +- 0.03 [0.17 - 0.24],0.32 +- 0.07 [0.24 - 0.42]
